# 📘 Домашнє завдання №23. Інструменти для роботи з LLM: LangChain та RAG

Виконав: **Bohdan Pinchuk**

Link: https://github.com/BogdanPinchuk/DataScience-PBY_HW24

## Завдання

Створити просту RAG-систему (Retrieval-Augmented Generation), яка відповідає на питання за текстом Конституції України:

https://unece.org/fileadmin/DAM/hlm/prgm/cph/experts/ukraine/ukr.constitution.e.pdf


### Що потрібно реалізувати

Вам необхідно побудувати систему, яка:

1. Завантажує PDF документ
2. Розбиває його на фрагменти (chunking)
3. Перетворює текст у embeddings
4. Зберігає їх у векторній базі даних (FAISS або Chroma)
5. Виконує semantic search по запиту користувача
6. Передає знайдений контекст у LLM
7. Генерує відповідь на основі Конституції України

### Рекомендований стек

Обробка документа:
- `PyPDFLoader` або аналог

Chunking:
- `RecursiveCharacterTextSplitter`

Embeddings:
- `sentence-transformers/all-MiniLM-L6-v2`
- або multilingual варіант:
  - `intfloat/multilingual-e5-base`


### Рекомендовані LLM (для генерації відповідей)

Простий варіант:
- `Qwen/Qwen2.5-1.5B-Instruct`

Кращі варіанти для україномовного RAG:
- `Qwen/Qwen2.5-7B-Instruct`

### Вимоги до реалізації

- Використати vector database (`FAISS` або `Chroma`)
- Реалізувати retrieval (`top-k` = 3–5 документів)
- Передати контекст у prompt
- Забезпечити відповідь виключно на основі документа

### Тестові питання

Перевірте вашу систему на таких запитах:

1. Які основні права і свободи людини гарантує Конституція України?

2. Яка структура державної влади описана в Конституції України?

3. Які умови зміни Конституції України передбачені в документі?

### Порада

Якщо плануєте україномовний RAG — використовуйте мультимовні embeddings та сучасні instruct-моделі, оскільки якість відповіді залежить не тільки від retrieval, але й від мовної моделі.

In [1]:
# Silent installation or update

# Clean cache
!python3 -m pip cache purge -q

# Force updating
package_update = [
    "pip",
    "scikit-learn",
    "pandas",
    "kagglehub",
    "kagglesdk",
]

for package_name in package_update:
    !bash -c "python3 -m pip install -U '{package_name}' -q"

# Install missing packages
package_array = [
    "jinja2",
    "ipywidgets",
    "nbformat",
    "kagglehub[pandas-datasets]",
    "numpy",
    "matplotlib",
    "scipy",
    "statsmodels",
    "joblib",
    "torch",
    "transformers",
]

for package_name in package_array:
    !bash -c "python3 -m pip show '{package_name}' > /dev/null 2>&1 || python3 -m pip install -U '{package_name}' -q"


In [2]:
# Synchronization with remote source

import shutil
from pathlib import Path

# Input data
hm_version = 24

# Solution
git_project_url = f"https://github.com/BogdanPinchuk/DataScience-PBY_HW{hm_version}.git"
main_file_name = f"Bohdan_Pinchuk_DS_HW{hm_version}.ipynb"

# upload all files
current_path = !pwd
current_path = current_path[0]
parent_path = !dirname "$current_path"
parent_path = parent_path[0]
temp_path = f"{parent_path}/temp"

# Clone data
!rm -rf "$temp_path"
!git clone "$git_project_url" "$temp_path"

source = Path(temp_path)
destination = Path(current_path)
exclude = {main_file_name, ".git", ".idea"}

for item in source.iterdir():
    if item.name in exclude:
        continue

    target = destination / item.name
    if item.is_dir():
        shutil.copytree(item, target, dirs_exist_ok=True)
    else:
        shutil.copy2(item, target)

# Clean temp folder
!rm -rf "$temp_path"

Cloning into '/Users/bohdanpinchuk/Documents/Data Science/Development/Data_Science/Practical_tasks/Homework_24/temp'...
remote: Enumerating objects: 13, done.
remote: Counting objects: 100% (13/13), done.
remote: Compressing objects: 100% (13/13), done.
remote: Total 13 (delta 0), reused 13 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (13/13), 6.44 KiB | 3.22 MiB/s, done.


# ✅ Крок: Завантажити PDF документ